In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Warnings ko hide karne ke liye taki output clean dikhe
warnings.filterwarnings('ignore')

# Tumhara diya hua exact file path
file_path = r"D:\UNIFIED MENTOR PROJECT\API DATA SET\APL_Logistics.csv"

# Dataset load kar rahe hain (encoding latin1 lagaya hai kyunki supply chain data me ajeeb characters ho sakte hain)
try:
    df = pd.read_csv(file_path, encoding='latin1')
    print("Data successfully load ho gaya hai!")
except Exception as e:
    print("Error aayi hai:", e)

# Basic Dataset details check karte hain
print("\nDataset Shape (Rows, Columns):", df.shape)

print("\nMissing Values Count (Sirf null wale columns):")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

# Starting ke 5 rows dekhne ke liye
df.head()

Data successfully load ho gaya hai!

Dataset Shape (Rows, Columns): (180519, 40)

Missing Values Count (Sirf null wale columns):
Customer Lname      8
Customer Zipcode    3
dtype: int64


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,5,499.95,472.45,159.69,South Asia,Maharashtra,COMPLETE,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,5,199.95,167.96,48.71,Central America,Cortés,ON_HOLD,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,1,199.99,181.99,87.36,Central America,Cortés,ON_HOLD,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,1,199.99,175.99,-41.89,East of USA,Nueva York,COMPLETE,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,1,50.00,40.00,10.00,East of USA,Nueva York,COMPLETE,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class


In [2]:
# Step 2: Data Cleaning aur Feature Preprocessing

# Faltu columns aur leakage wale columns ki list jo humein hatani hai
columns_to_drop = [
    'Customer Fname', 'Customer Lname', 'Customer Street', 'Customer Zipcode',
    'Order Customer Id', 'Order City', 'Order State', 'Customer City', 
    'Product Name', 'Category Id', 'Department Id', 'Customer Id',
    'Latitude', 'Longitude', 'Days for shipping (real)', 'Delivery Status'
]

# In columns ko hatate hain (errors='ignore' lagaya hai taki agar koi column na mile toh error na aaye)
df_clean = df.drop(columns=columns_to_drop, errors='ignore')

# Agar thodi bahut missing values bachi hain, toh un rows ko hata dete hain (safe aur simple approach)
df_clean = df_clean.dropna()

# Target variable (jo humein predict karna hai)
target = 'Late_delivery_risk'

# X (Features/Inputs) aur y (Target/Output) ko alag karte hain
y = df_clean[target]
X = df_clean.drop(columns=[target])

# Categorical (text) data ko numbers me badalna (One-Hot Encoding)
# drop_first=True lagane se dummy variable trap se bachte hain (standard practice)
X_encoded = pd.get_dummies(X, drop_first=True)

print("Data Cleaning aur Encoding complete ho gayi hai!")
print("Features (X) ka naya shape:", X_encoded.shape)

# Clean kiye hue data ka preview
X_encoded.head()

Data Cleaning aur Encoding complete ho gayi hai!
Features (X) ka naya shape: (180519, 322)


,Days for shipment (scheduled),Benefit per order,Sales per customer,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,...,Order Status_COMPLETE,Order Status_ON_HOLD,Order Status_PAYMENT_REVIEW,Order Status_PENDING,Order Status_PENDING_PAYMENT,Order Status_PROCESSING,Order Status_SUSPECTED_FRAUD,Shipping Mode_Same Day,Shipping Mode_Second Class,Shipping Mode_Standard Class
0,4,159.69,472.45,27.50,0.06,99.99,0.34,5,499.95,472.45,...,True,False,False,False,False,False,False,False,False,True
1,4,48.71,167.96,31.99,0.16,39.99,0.29,5,199.95,167.96,...,False,True,False,False,False,False,False,False,False,True
2,4,87.36,181.99,18.00,0.09,199.99,0.48,1,199.99,181.99,...,False,True,False,False,False,False,False,False,False,True
3,4,-41.89,175.99,24.00,0.12,199.99,-0.24,1,199.99,175.99,...,True,False,False,False,False,False,False,False,False,True
4,4,10.00,40.00,10.00,0.20,50.00,0.25,1,50.00,40.00,...,True,False,False,False,False,False,False,False,False,True


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

# 1. Data ko train aur test me divide karna (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 2. Model initialize karna (Random Forest intermediate projects ke liye standard hai)
# n_estimators=100 ka matlab hum 100 decision trees ka use kar rahe hain
model = RandomForestClassifier(n_estimators=100, random_state=42)

print("Model train ho raha hai, isme thoda time lag sakta hai...")

# Model ko train karna
model.fit(X_train, y_train)

# 3. Test data par prediction karna aur Accuracy check karna
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Training Complete!")
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Pura classification report dekhne ke liye (Precision, Recall etc.)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 4. Model aur Columns ko save karna (Streamlit me kaam aayega)
# Model ko save kar rahe hain
with open('delivery_risk_model.pkl', 'wb') as file:
    pickle.dump(model, file)

# Columns list ko bhi save kar rahe hain taki naye data ka format same rahe
with open('model_columns.pkl', 'wb') as file:
    pickle.dump(list(X_encoded.columns), file)

print("\nSuccess: 'delivery_risk_model.pkl' aur 'model_columns.pkl' files save ho gayi hain!")

Model train ho raha hai, isme thoda time lag sakta hai...

Model Training Complete!
Model Accuracy: 71.39%

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.79      0.71     16384
           1       0.79      0.65      0.71     19720

    accuracy                           0.71     36104
   macro avg       0.72      0.72      0.71     36104
weighted avg       0.73      0.71      0.71     36104


Success: 'delivery_risk_model.pkl' aur 'model_columns.pkl' files save ho gayi hain!
